# Lab 22 — Final Research Capstone

*AI Cybersecurity Masterclass · Lab 22 · AI60secLab*

**Companion video:** [▶ What Is Experimental Design in Cybersecurity ML?](https://www.youtube.com/watch?v=cqDkcR_TJp8&list=PLG4l6Ssi9A00) · [▶ From Security Model to Published Research](https://www.youtube.com/watch?v=cUUY_uEApcE&list=PLG4l6Ssi9A00) · [▶ Turn Security AI Into Decisions, Not Just Alerts](https://www.youtube.com/watch?v=O1Rr-mCNrWY&list=PLG4l6Ssi9A00) · [▶ How to Finalize a Cybersecurity Research Paper in 60 Seconds](https://www.youtube.com/watch?v=4Uby8PYnCGY&list=PLG4l6Ssi9A00)
## Adaptive Cyber Threat Detection Under Shift and Adversarial Conditions

**Core research question:** How does a machine-learning cyber threat detector behave when the training data, input distribution, or decision environment changes, and which defensive intervention best preserves useful detection performance?

### Experimental design

Each condition changes **one factor** relative to a named baseline:

| Condition | What changes | Compared with |
|---|---|---|
| C0 | Baseline model, default 0.5 threshold | (reference) |
| C1 | Threshold chosen on validation data by a fixed rule (max F2) | C0 |
| C2 | C1 evaluated on bounded input stress | C1 |
| C3 | 5% label-poisoned training, same threshold **rule** | C1 |
| C4 | Noise-augmented training, same threshold **rule**, clean and stressed | C1 and C2 |

Rules that keep the comparison fair:
- same data, preprocessing, metrics, and model settings everywhere;
- lock the threshold **rule**, not the number: each retrained model gets its own threshold from the same validation procedure;
- repeat with several seeds and report the spread;
- a delta only counts if it is bigger than the spread between runs;
- save one row per result.

## Security foundation: security research and responsible reporting

Security research has its own ethics and norms:
- **controlled environments:** test on data and systems you own or are authorized to use;
- **responsible disclosure:** if you find a real vulnerability, report it privately to the owner and give them time to fix it before publishing;
- **honest claims:** state the threat model, the data, and the limits. Security decisions are made on your claims, so overstating them causes real harm.

Writing for decision makers matters too. A CISO needs the conclusion first, then the evidence, the risk, and what to do next. A wall of metrics is not decision support.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import logistic_pipeline
from src.evaluation import binary_metrics
from src.thresholds import (
    threshold_sweep,
    choose_threshold,
    apply_threshold,
)
from src.drift import domain_classifier_auc
from src.adversarial import (
    available_numeric_attack_features,
    training_bounds,
    bounded_random_perturbation,
    augment_attacks_with_noise,
)
from src.poisoning import random_label_flip

train, test = load_unsw(DATA_DIR)

X_full, y_full = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_full, X_test = align_columns(X_full, X_test)

X_full = drop_identifier_like_columns(X_full)
X_test = drop_identifier_like_columns(X_test)

stress_columns = available_numeric_attack_features(X_full)
bounds = training_bounds(X_full, stress_columns)

SEEDS = [0, 1, 2]

In [3]:
def fit_with_threshold_rule(X_tr, y_tr, X_va, y_va):
    """Train a model and pick its threshold with the SAME validation rule."""
    model = logistic_pipeline(X_tr)
    model.fit(X_tr, y_tr)
    sweep = threshold_sweep(y_va, model.predict_proba(X_va)[:, 1], beta=2.0)
    threshold, _ = choose_threshold(sweep, objective="f_beta")
    return model, threshold


def result_row(condition, seed, model, X_eval, threshold):
    scores = model.predict_proba(X_eval)[:, 1]
    m = binary_metrics(y_test, apply_threshold(scores, threshold), scores)
    return {
        "condition": condition,
        "seed": seed,
        "threshold": threshold,
        "precision": m.precision,
        "recall": m.recall,
        "f1": m.f1,
        "false_negative_rate": m.false_negative_rate,
        "roc_auc": m.roc_auc,
    }


rows = []
for seed in SEEDS:
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_full, y_full, test_size=0.20, stratify=y_full, random_state=seed,
    )

    baseline, tuned_threshold = fit_with_threshold_rule(X_train, y_train, X_valid, y_valid)
    X_stressed = bounded_random_perturbation(
        X_test, columns=stress_columns, epsilon=0.10, bounds=bounds, random_state=seed,
    )

    rows.append(result_row("C0 baseline, threshold 0.5", seed, baseline, X_test, 0.5))
    rows.append(result_row("C1 validation-selected threshold", seed, baseline, X_test, tuned_threshold))
    rows.append(result_row("C2 C1 under input stress", seed, baseline, X_stressed, tuned_threshold))

    poisoned_y = random_label_flip(y_train, fraction=0.05, random_state=seed)
    poisoned, poisoned_threshold = fit_with_threshold_rule(
        X_train.reset_index(drop=True), poisoned_y, X_valid, y_valid,
    )
    rows.append(result_row("C3 5% label poisoning", seed, poisoned, X_test, poisoned_threshold))

    X_aug, y_aug = augment_attacks_with_noise(
        X_train, y_train, columns=stress_columns, epsilon=0.05, copies=1,
        bounds=bounds, random_state=seed,
    )
    augmented, augmented_threshold = fit_with_threshold_rule(X_aug, y_aug, X_valid, y_valid)
    rows.append(result_row("C4 augmented, clean test", seed, augmented, X_test, augmented_threshold))
    rows.append(result_row("C4 augmented, under stress", seed, augmented, X_stressed, augmented_threshold))

runs = pd.DataFrame(rows)
runs.to_csv(RESULTS_DIR / "22_capstone_runs.csv", index=False)
print("Saved", len(runs), "rows to results/22_capstone_runs.csv")

Saved 18 rows to results/22_capstone_runs.csv


In [4]:
domain_auc, _ = domain_classifier_auc(
    X_full,
    X_test,
    sample_per_domain=15000,
    random_state=42,
)

print(
    "Train-vs-test domain classifier AUC:",
    round(domain_auc, 4),
)

Train-vs-test domain classifier AUC: 0.6629


In [5]:
summary = (
    runs.groupby("condition", sort=False)[["threshold", "precision", "recall", "f1", "false_negative_rate"]]
    .agg(["mean", "std"])
)
display(summary.round(4))

threshold         precision          recall  \
                                      mean     std      mean     std    mean   
condition                                                                      
C0 baseline, threshold 0.5          0.5000  0.0000    0.8015  0.0016  0.9313   
C1 validation-selected threshold    0.0500  0.0000    0.7435  0.0011  0.9986   
C2 C1 under input stress            0.0500  0.0000    0.7435  0.0012  0.9986   
C3 5% label poisoning               0.1767  0.0551    0.7433  0.0008  0.9986   
C4 augmented, clean test            0.0500  0.0000    0.7435  0.0006  0.9985   
C4 augmented, under stress          0.0500  0.0000    0.7435  0.0006  0.9985   

                                              f1         false_negative_rate  \
                                     std    mean     std                mean   
condition                                                                      
C0 baseline, threshold 0.5        0.0014  0.8615  0.0006              0.0687   
C1 validation-selected threshold  0.0001  0.8524  0.0007              0.0014   
C2 C1 under input stress          0.0001  0.8524  0.0007              0.0014   
C3 5% label poisoning             0.0005  0.8522  0.0004              0.0014   
C4 augmented, clean test          0.0001  0.8524  0.0004              0.0015   
C4 augmented, under stress        0.0002  0.8523  0.0005              0.0015   

                                          
                                     std  
condition                                 
C0 baseline, threshold 0.5        0.0014  
C1 validation-selected threshold  0.0001  
C2 C1 under input stress          0.0001  
C3 5% label poisoning             0.0005  
C4 augmented, clean test          0.0001  
C4 augmented, under stress        0.0002

## Which deltas count?

Each condition is compared with the condition it changes one factor from. A delta counts only if it is more than twice the largest seed standard deviation of the two conditions being compared.

In [6]:
comparisons = [
    ("C1 validation-selected threshold", "C0 baseline, threshold 0.5"),
    ("C2 C1 under input stress", "C1 validation-selected threshold"),
    ("C3 5% label poisoning", "C1 validation-selected threshold"),
    ("C4 augmented, clean test", "C1 validation-selected threshold"),
    ("C4 augmented, under stress", "C2 C1 under input stress"),
]

stats = runs.groupby("condition")[["recall", "f1"]].agg(["mean", "std"])
delta_rows = []
for treatment, reference in comparisons:
    for metric in ["recall", "f1"]:
        delta = stats.loc[treatment, (metric, "mean")] - stats.loc[reference, (metric, "mean")]
        spread = max(stats.loc[treatment, (metric, "std")], stats.loc[reference, (metric, "std")])
        delta_rows.append({
            "treatment": treatment,
            "reference": reference,
            "metric": metric,
            "delta": delta,
            "seed_spread": spread,
            "counts": abs(delta) > 2 * spread,
        })

deltas = pd.DataFrame(delta_rows)
display(deltas.round(4))

,treatment,reference,metric,delta,seed_spread,counts
0,C1 validation-selected threshold,"C0 baseline, threshold 0.5",recall,0.0673,0.0014,True
1,C1 validation-selected threshold,"C0 baseline, threshold 0.5",f1,-0.0091,0.0007,True
2,C2 C1 under input stress,C1 validation-selected threshold,recall,-0.0000,0.0001,False
3,C2 C1 under input stress,C1 validation-selected threshold,f1,-0.0000,0.0007,False
4,C3 5% label poisoning,C1 validation-selected threshold,recall,0.0000,0.0005,False
5,C3 5% label poisoning,C1 validation-selected threshold,f1,-0.0001,0.0007,False
6,"C4 augmented, clean test",C1 validation-selected threshold,recall,-0.0001,0.0001,False
7,"C4 augmented, clean test",C1 validation-selected threshold,f1,-0.0000,0.0007,False
8,"C4 augmented, under stress",C2 C1 under input stress,recall,-0.0001,0.0002,False
9,"C4 augmented, under stress",C2 C1 under input stress,f1,-0.0000,0.0007,False


In [7]:
print(
    "Independent distribution-shift signal (domain classifier AUC):",
    round(domain_auc, 4),
)
print("Conditions whose deltas exceed the seed spread:")
display(deltas[deltas["counts"]][["treatment", "metric", "delta"]].round(4))

Independent distribution-shift signal (domain classifier AUC): 0.6629
Conditions whose deltas exceed the seed spread:


,treatment,metric,delta
0,C1 validation-selected threshold,recall,0.0673
1,C1 validation-selected threshold,f1,-0.0091


# Final white-paper structure

1. **Abstract**
2. **Research question and falsifiable hypotheses**
3. **Dataset and limitations**
4. **Methods and train/validation/test separation**
5. **Clean results**
6. **Shift, stress, and poisoning results**
7. **Mitigation comparison**
8. **Security interpretation**
9. **Limitations**
10. **Reproducibility and future research**

**Final principle:** do not claim “the model is secure.” State exactly what conditions were tested, what degraded, what improved, and what remains unknown.

## Decision report template (the professional deliverable)

The white paper above is for reviewers. Security leaders need a shorter **decision report**: conclusion first, evidence second, framework mapping third, and a clear ask. Use this structure, which mirrors how professional red-team and assessment reports are written.

1. **Bottom line up front (BLUF):** one or two sentences a CISO can act on.
2. **Scope and threat model:** what was tested, what was not, and which adversary you assumed.
3. **Findings table:** each finding with severity, evidence, and the condition that produced it.
4. **Framework mapping:**
   - **MITRE ATT&CK** for the attacker behaviors the detector is meant to see (Lab 18);
   - **MITRE ATLAS** for attacks on the ML system itself, such as poisoning and evasion (Lab 19);
   - **OWASP Top 10 for LLM Applications (2025)** if an LLM component is in scope (Unit 5);
   - **NIST AI Risk Management Framework** functions: **Govern, Map, Measure, Manage**. This capstone is mostly *Measure* evidence.
5. **Limitations:** what the evidence does not prove.
6. **Recommendations:** each with an owner and a decision, as in Lab 29.
7. **Disclosure note:** if you found a real flaw in someone else's product, how and when you reported it.

The next cell drafts the report from this lab's own results. Edit the wording; keep the numbers.

In [ ]:
def fmt(x):
    return f"{x:.3f}"

counted = deltas[deltas["counts"]]
not_counted = deltas[~deltas["counts"]]
c1 = stats.loc["C1 validation-selected threshold"]
c0 = stats.loc["C0 baseline, threshold 0.5"]

findings = []
for row in deltas.itertuples():
    severity = "High" if (row.counts and row.delta < 0) else ("Info" if not row.counts else "Positive")
    findings.append(
        f"| {row.treatment} vs {row.reference} | {row.metric} | {row.delta:+.3f} | "
        f"{row.seed_spread:.3f} | {'yes' if row.counts else 'no'} | {severity} |"
    )

report = f"""# Decision report: ML threat detector under shift and attack

## Bottom line up front
Choosing the alert threshold on validation data changed recall by {c1[("recall","mean")] - c0[("recall","mean")]:+.3f}.
{len(counted)} of {len(deltas)} tested deltas exceeded the seed spread; the rest are indistinguishable from noise.
Recommendation: prioritize threshold governance and monitoring before investing in robustness retraining.

## Scope and threat model
- Detector: logistic regression on the UNSW-NB15 prepared partition (synthetic testbed traffic, 2015).
- Conditions: bounded random input stress (epsilon 0.10), 5% random label poisoning, noise augmentation.
- Adversary assumed: undirected noise and random label corruption. Targeted or traffic-valid attacks were NOT tested.
- Seeds: {len(SEEDS)}. Train/test domain-classifier AUC: {fmt(domain_auc)}.

## Findings
| Comparison | Metric | Delta | Seed spread | Counts | Severity |
|---|---|---|---|---|---|
{chr(10).join(findings)}

## Framework mapping
- MITRE ATLAS: data poisoning (C3) and evasion-style input perturbation (C2, C4).
- MITRE ATT&CK: the detector targets network-visible behaviors such as reconnaissance and impact; flow data cannot confirm techniques.
- NIST AI RMF: Measure (robustness and validity evidence); Manage (threshold governance recommendation).

## Limitations
- One synthetic dataset with known artifacts (Lab 00b); results may not transfer.
- Random perturbations measure sensitivity, not resistance to a real adversary.
- {len(not_counted)} deltas were within noise; absence of an effect here is not proof of robustness.

## Recommendations
| Action | Owner | Decision |
|---|---|---|
| Re-select thresholds on validation data with a documented rule | Detection engineering | Adopt before next release |
| Monitor score drift and alert volume against a baseline | Security monitoring | Implement (Lab 28 pattern) |
| Test targeted, traffic-valid evasion before claiming robustness | ML engineering | Schedule next quarter |
"""

report_path = RESULTS_DIR / "22_decision_report.md"
report_path.write_text(report)
print("Saved:", report_path)
print(report)

## From lab to SOC

This capstone found that the threshold rule mattered more than the defenses tested. A good security report says that plainly, explains why, and recommends where to spend effort, even when the answer is less exciting than "we built a robust model".

**Check yourself**
1. Write the first sentence of a report to a CISO based on this capstone's results.
2. You find a real flaw in a vendor's detection product. What do you do first?
3. Why is a negative result ("no effect beyond seed spread") valuable to a security team?